# 01. Profile Image 3-Way Classification

Uses `pipeline.run_collect()` to execute the full pipeline:
**sample → enrich → prefilter → download → classify (YOLOv8 + CLIP)**.

All steps are resumable — re-running any cell continues from the last checkpoint.

| Category | Method |
|----------|--------|
| **Default** | URL + pixel prefilter (Gravatar, identicon) |
| **Anime**   | YOLOv8 anime-face detection **AND** CLIP zero-shot (anime vs human) |
| **Photo**   | Everything else |


## 0. Setup

In [ ]:
TOTAL_USERS = 1000

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from pipeline import (
    PipelineConfig,
    run_collect,
    print_status,
)

DATA_DIR = Path('../data').resolve()
YOLO_MODEL = Path('../yolov8x6_animeface.pt').resolve()

cfg = PipelineConfig(
    data_dir=DATA_DIR,
    yolo_model_path=YOLO_MODEL,
    total=TOTAL_USERS,
    clip_model_name='ViT-B-32',
    clip_pretrained='laion2b_s34b_b79k',
)
print(f'data_dir: {cfg.data_dir}')
print(f'yolo    : {cfg.yolo_model_path} (exists={cfg.yolo_model_path.exists()})')


## A. One-shot — run everything

Run this single cell for the full pipeline. Resumes automatically.

In [ ]:
await run_collect(cfg)

## Status snapshot

In [ ]:
print_status(cfg.data_dir)

## Results — category counts

In [ ]:
df = pd.read_csv(cfg.data_dir / 'processed' / 'classified_3cat.csv')
print(df['profile_type'].value_counts())
df.head()

## Sample images per category

In [ ]:
AVATAR_DIR = cfg.data_dir / 'raw' / 'avatars'


def plot_samples(sample_df, title):
    n = len(sample_df)
    if n == 0:
        print(f'(no samples for: {title})')
        return
    n_cols = 5
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, (_, row) in enumerate(sample_df.iterrows()):
        uid = row['uid']
        img_path = AVATAR_DIR / f'{uid}.png'
        try:
            img = Image.open(img_path)
        except Exception:
            img = Image.new('RGB', (256, 256), color='gray')
        axes[i].imshow(img)
        axes[i].axis('off')
        caption = f"{uid}\nyolo={row['anime_conf']:.2f}  clip={row['clip_anime_score']:.2f}"
        axes[i].set_title(caption, fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


anime_df = df[df['profile_type'] == 'Anime']
photo_df = df[df['profile_type'] == 'Photo']
print(f"Anime: {len(anime_df)}, Photo: {len(photo_df)}")

plot_samples(anime_df.sample(n=min(10, len(anime_df)), random_state=42),
             'Anime (YOLO + CLIP confirmed)')
plot_samples(photo_df[photo_df['is_anime']].sample(
    n=min(10, (photo_df['is_anime']).sum()), random_state=42),
             'YOLO face but CLIP → human (filtered out as Photo)')
plot_samples(photo_df[~photo_df['is_anime']].sample(
    n=min(10, (~photo_df['is_anime']).sum()), random_state=42),
             'No face (Photo)')


## YOLO confidence distribution (Anime only)

In [ ]:
anime_confs = df.loc[df['profile_type'] == 'Anime', 'anime_conf']

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(anime_confs, bins=30, edgecolor='black', alpha=0.7, color='#ff6b6b')
ax.set_xlabel('YOLOv8 Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Confirmed Anime PFP — YOLO Confidence')
plt.tight_layout()
plt.show()
if len(anime_confs):
    print(f'mean={anime_confs.mean():.3f}  median={anime_confs.median():.3f}  '
          f'min={anime_confs.min():.3f}  max={anime_confs.max():.3f}')
